# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

import os
os.environ["PATH"] += os.pathsep + "/home/ugoyal/.local/bin"

# Create a virtual environment
!uv venv .venv --seed --clear

# Install dependencies
!.venv/bin/python -m pip install \
    "numpy<2" \
    "torch==2.2.2" \
    "transformers>=4.51,<4.57" \
    "tokenizers>=0.21,<0.22" \
    "accelerate==0.30.1" \
    sympy tqdm antlr4-python3-runtime==4.11.1 ipykernel jupyter sentencepiece \
    "bitsandbytes"

# Install Jupyter kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Then select: Python (cse151b)")

downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /home/ugoyal/.local/bin
  uv
  uvx
everything's installed!
Using CPython 3.11.9 interpreter at: /opt/conda/bin/python3
Creating virtual environment with seed packages at: .venv
 + packaging==26.1
 + pip==26.0.1
 + setuptools==82.0.1
 + wheel==0.46.3
Activate with: source .venv/bin/activate
  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached torch-2.2.2-cp311-cp311-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached transformers-4.56.2-py3-none-any.whl.metadata (40 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
  Using cached accelerate-0.30.1-py3-none-any.whl.metadata (18 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached antlr4_python3_runtime-4.11.1-py3-none-any.whl.metadata (291 bytes)
  Using cached i

### Run the cell below every time to activate the installed environment. 

In [1]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
#MAX_TOKENS  = 4096
MAX_TOKENS  = 500

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 3.1 Create Embeddings and Cluster

In [13]:
pip install -U sentence-transformers

  Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 10.4 MB/s  0:00:00
Using cached scikit_learn-1.8.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


In [14]:
# SBERT Embedding 

from sentence_transformers import SentenceTransformer

# Load model and define sentences
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = []

# Look at all questions and put them in texts
for item in data: 
    q = item["question"]
    options = item.get("options")

    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = " ".join(f"{lbl}. {opt}" for lbl, opt in zip(labels, options))
        text = f"Question: {q} Options: {opts_text}"
    else:
        text = f"Question: {q}"
    
    texts.append(text)

# Encode all at once (fast + efficient)
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)  
# (1126, 384) - 1126 questions with a 384 dimension vector

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

(1126, 384)


In [17]:
# KMean Clustering
from sklearn.cluster import KMeans
import numpy as np

k = 8

kmeans = KMeans(n_clusters=k, random_state=23, n_init=10)
cluster_ids = kmeans.fit_predict(embeddings)

print(cluster_ids.shape)       
print(kmeans.cluster_centers_.shape) 

clusters = {}
for cid in range(k):
    clusters[cid] = {
        "center": kmeans.cluster_centers_[cid],
        "points": []
    }

for idx, cid in enumerate(cluster_ids):
    clusters[cid]["points"].append(idx)

print("Number of clusters:", len(clusters))
for cid in clusters:
    print(f"Cluster {cid}: {len(clusters[cid]['points'])} questions")

(1126,)
(8, 384)
Number of clusters: 8
Cluster 0: 156 questions
Cluster 1: 207 questions
Cluster 2: 198 questions
Cluster 3: 153 questions
Cluster 4: 136 questions
Cluster 5: 112 questions
Cluster 6: 124 questions
Cluster 7: 40 questions


In [20]:
import numpy as np

representatives_per_cluster = 2
cluster_rep_indices = {}
cluster_reps = {}

for cid in range(k):
    # indices of question in cluster
    cluster_idxs = np.where(cluster_ids == cid)[0]

    # embeddings of question in cluster
    cluster_embeds = embeddings[cluster_idxs]

    # centroid of the cluster
    centroid = kmeans.cluster_centers_[cid]

    # dist of each point to center
    dists = np.linalg.norm(cluster_embeds - centroid, axis=1)

    # pick 2 clistest point
    sorted_local = np.argsort(dists)[:representatives_per_cluster]

    # ma
    chosen_global = cluster_idxs[sorted_local]
    
    cluster_rep_indices[cid] = chosen_global.tolist()
    cluster_reps[cid] = [data[idx] for idx in chosen_global]

print(cluster_rep_indices)

{0: [111, 957], 1: [661, 422], 2: [263, 811], 3: [467, 1098], 4: [70, 721], 5: [402, 10], 6: [310, 638], 7: [1045, 499]}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [48]:
SYSTEM_PROMPT_MATH_REPRESENTATIVES = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Show the key reasoning clearly and concisely. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ_REPRESENTATIVES = (
    "You are an expert mathematician. Solve the problem step-by-step by analyzing the answer choices carefully. "
    "Eliminate incorrect option when helpful. "
    "Select the single best answer. "
    "Output ONLY the letter of your chosen best option inside \\boxed{}, e.g. \\boxed{C}."
)

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem carefully step-by-step. "
    "Put your final answer inside \\boxed{}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Solve the problem carefully. "
    "At the end, output the SINGLE best answer inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

def build_fewshot_prompt(item, exemplars):
    """
    item: current question
    exemplars: list of representative examples (with solutions)
    """
    
    parts = []

    # Add exemplars
    for ex in exemplars:
        q = ex["question"]
        options = ex.get("options")
        sol = ex["solution"]   # generated CoT
        
        if options:
            labels = [chr(65 + i) for i in range(len(options))]
            opts_text = "\n".join(
                f"{lbl}. {opt.strip()}" 
                for lbl, opt in zip(labels, options)
            )
            
            block = (
                f"Example Problem:\n{q}\n\n"
                f"Options:\n{opts_text}\n\n"
                f"Step-by-step Solution:\n{sol}\n"
            )
        else:
            block = (
                f"Example Problem:\n{q}\n\n"
                f"Step-by-step Solution:\n{sol}\n"
            )
        
        parts.append(block)

    # Add target question
    q = item["question"]
    opts = item.get("options")
    
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        options_text = "\n".join(
            f"{lbl}. {opt.strip()}" 
            for lbl, opt in zip(labels, options)
        )
        
        target = (
            f"Now solve this new problem in the same style.\n\n"
            f"Problem:\n{q}\n\nOptions:\n{opts_text}"
        )
    else:
        target = (
            f"Now solve this new problem in the same style.\n\n"
            f"Problem:\n{q}"
        )

    parts.append(target)

    user_prompt = "\n\n".join(parts)

    # Choose system prompt
    if item.get("options"):
        system_prompt = SYSTEM_PROMPT_MCQ
    else:
        system_prompt = SYSTEM_PROMPT_MATH

    return system_prompt, user_prompt

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [25]:
import gc

# Delete model and tokenizer if they exist
try:
    del model
    del tokenizer
except:
    pass

# Clear Python garbage
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Reset cached memory stats
torch.cuda.reset_peak_memory_stats()

print("Memory cleared!")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

Memory cleared!
Allocated: 0.01 GB
Cached: 0.05 GB


In [32]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side='left',
)

In [36]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

def generate_batch(
    prompts: list[str],
    max_new_tokens: int = MAX_TOKENS,
    do_sample: bool = True,
    temperature: float = 0.6,
    top_p: float = 0.95,
    top_k: int = 20,
    batch_size: int = 1,
):
    all_responses = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(DEVICE)

        gen_kwargs = dict(
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        if do_sample:
            gen_kwargs.update(
                temperature=temperature,
                top_p=top_p,
                top_k=top_k,
            )

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        input_len = inputs["input_ids"].shape[1]

        for output in outputs:
            generated_ids = output[input_len:]
            all_responses.append(
                tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            )

        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return all_responses

print(f"Model loaded on {DEVICE}.")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on cuda.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [37]:
rep_prompts = []
rep_items = []

# Go through ALL representatives
for cid, rep_idxs in cluster_rep_indices.items():
    for idx in rep_idxs:
        item = data[idx]

        system, user = build_prompt(item["question"], item.get("options"))

        prompt_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user",   "content": user}],
            tokenize=False,
            add_generation_prompt=True,
        )

        rep_prompts.append(prompt_text)
        rep_items.append(item)

print(f"Generating representative solutions for {len(rep_prompts)} examples...")

rep_outputs = generate_batch(
    rep_prompts,
    max_new_tokens=200,
    do_sample=False, # more stable reasoning
    batch_size=1
)

# Store them
rep_solutions = {}
for item, output in zip(rep_items, rep_outputs):
    rep_solutions[item["id"]] = output

print("Representative solutions generated!")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating representative solutions for 16 examples...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Representative solutions generated!


In [53]:
selected_items = data[:]
prompts = []

for item, cid in zip(data, cluster_ids):
    item["cluster_id"] = int(cid)
    
for item in selected_items:
    cid = item["cluster_id"]

    rep_idxs = cluster_rep_indices[cid]

    exemplars = []
    for idx in rep_idxs:
        ex_item = data[idx]

        if ex_item["id"] == item["id"]:
            continue

        exemplars.append({
            "question": ex_item["question"],
            "options": ex_item.get("options"),
            "solution": rep_solutions[ex_item["id"]]
        })

    system, user = build_fewshot_prompt(item, exemplars)

    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions...")
responses = []

for i, prompt in enumerate(prompts):
    output = generate_batch([prompt])[0]
    responses.append(output)

    # save every step
    with open("partial_results.jsonl", "a") as f:
        f.write(json.dumps({
            "id": selected_items[i]["id"],
            "response": output
        }) + "\n")

    if i % 10 == 0:
        print(f"Done {i}/{len(prompts)}")

for item, response in zip(selected_items, responses):
    print(f"\n── Response (id={item.get('id')}) ──")
    print(response[:400], "..." if len(response) > 400 else "")

Generating responses for 1126 questions...
Done 0/1126
Done 10/1126


KeyboardInterrupt: 

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [54]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   2%|▏         | 17/1126 [00:00<00:32, 34.16it/s]

Scoring complete. 17 results.


## 8. Summary

Print accuracy broken down by question type.

In [55]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    7  (14.29%)
  Free-form  :    1 /   10  (10.00%)
  Overall    :    2 /   17  (11.76%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [23]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 2 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!